In [3]:
%pip install pandas numpy gurobipy

  Using cached pandas-3.0.1-cp313-cp313-win_amd64.whl.metadata (19 kB)
  Using cached tzdata-2025.3-py2.py3-none-any.whl.metadata (1.4 kB)
Using cached pandas-3.0.1-cp313-cp313-win_amd64.whl (9.7 MB)
   ---------------------------------------- 0.0/12.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/12.3 MB ? eta -:--:--
    --------------------------------------- 0.3/12.3 MB ? eta -:--:--
   -- ------------------------------------- 0.8/12.3 MB 1.4 MB/s eta 0:00:09
   --- ------------------------------------ 1.0/12.3 MB 1.5 MB/s eta 0:00:08
   ----- ---------------------------------- 1.6/12.3 MB 1.7 MB/s eta 0:00:07
   ----- ---------------------------------- 1.8/12.3 MB 1.7 MB/s eta 0:00:07
   -------- ------------------------------- 2.6/12.3 MB 1.9 MB/s eta 0:00:06
   --------- ------------------------------ 2.9/12.3 MB 1.9 MB/s eta 0:00:05
   ----------- ---------------------------- 3.4/12.3 MB 2.0 MB/s eta 0:00:05
   ------------ --------------------------- 3.9/12.

In [6]:
%pip install openpyxl

  Using cached openpyxl-3.1.5-py2.py3-none-any.whl.metadata (2.5 kB)
  Using cached et_xmlfile-2.0.0-py3-none-any.whl.metadata (2.7 kB)
Using cached openpyxl-3.1.5-py2.py3-none-any.whl (250 kB)
Using cached et_xmlfile-2.0.0-py3-none-any.whl (18 kB)

   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------

In [4]:
#Import Packages
import pandas as pd
import numpy as np
import gurobipy as gp
from gurobipy import GRB

In [7]:
# Load data
file_path = "Data Collection.xlsx"
df = pd.read_excel(file_path, sheet_name="Sheet1").copy()

# Clean column names
df.columns = (
    df.columns
    .str.strip()
    .str.replace(" ", "_")
    .str.replace("(", "", regex=False)
    .str.replace(")", "", regex=False)
    .str.replace("/", "_", regex=False)
)

# Rename a few columns to make coding easier
df = df.rename(columns={
    "Individual_Price": "price",
    "EnergyCalories": "calories",
    "Fat_g": "fat",
    "Saturates_Fat_g": "sat_fat",
    "Carbohydrates_g": "carbs",
    "Sugars_g": "sugar",
    "Fibre_g": "fibre",
    "Protein_g": "protein",
    "Salt_g": "salt",
    "Prime5.5GBP": "is_prime",
    "Category": "category",
    "Item": "item",
    "Vegetarian": "vegetarian",
    "Vegan": "vegan",
    "Dairy": "dairy",
    "Gluten": "gluten",
    "Porks": "pork",
    "Beef": "beef",
    "Chicken": "chicken",
    "Fish": "fish"
})

# Standardize category labels
df["category"] = df["category"].astype(str).str.strip().str.title()

# Convert numeric columns
numeric_cols = [
    "price", "calories", "fat", "sat_fat", "carbs", "sugar",
    "fibre", "protein", "salt"
]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)

# Convert binary columns
binary_cols = [
    "is_prime", "vegetarian", "vegan", "dairy", "gluten",
    "pork", "beef", "chicken", "fish"
]

for col in binary_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0).astype(int)

# Split into the three meal-deal categories
mains = df[df["category"] == "Main"].reset_index(drop=True)
snacks = df[df["category"] == "Snack"].reset_index(drop=True)
drinks = df[df["category"] == "Drink"].reset_index(drop=True)

print("Number of mains:", len(mains))
print("Number of snacks:", len(snacks))
print("Number of drinks:", len(drinks))

display(df.head())

Number of mains: 94
Number of snacks: 46
Number of drinks: 31


,item,category,price,calories,fat,sat_fat,carbs,sugar,fibre,protein,...,is_prime,vegetarian,vegan,dairy,Caffeine,gluten,pork,beef,chicken,fish
0,Tony's Chocolonely Milk Caramel Seasalt 47g,Snack,1.35,252.0,15.0,9.0,25.0,24.0,0.0,3.3,...,0,1,0,1,0,1,0,0,0,0
1,Pepsi regular soft drink 500ml,Drink,1.95,92.0,0.0,0.0,22.0,22.0,0.0,0.0,...,0,1,1,0,1,0,0,0,0,0
2,Tesco Fully Loaded Korean Style Pulled Pork Pr...,Main,4.50,669.0,29.3,11.3,69.8,19.3,3.0,30.3,...,0,0,0,1,0,1,1,0,0,0
3,Trek Protein Flapjack with Lotus Biscoff 50g,Snack,1.85,230.0,11.0,4.4,23.0,12.0,2.2,9.8,...,0,1,1,0,0,1,0,0,0,0
4,Tesco Chicken Tikka & Mango Chutney Sandwich,Main,2.75,333.0,4.6,1.0,49.0,8.6,4.2,21.9,...,0,0,0,1,0,1,0,0,1,0


In [8]:
#Define user-adjustable inputs

params = {
    # choose ONE objective each time
    "objective": "max_savings",  
    # options later can be:
    # "max_protein", "min_sugar", "min_fat", "min_carbs",
    # "min_calories", "min_salt", "max_savings"

    # Tesco meal deal prices
    "regular_price": 3.85,
    "prime_price": 5.50,

    # dietary constraints
    "vegetarian_only": False,
    "vegan_only": False,
    "no_pork": False,
    "no_beef": False,
    "no_chicken": False,
    "no_fish": False,
    "gluten_free": False,
    "dairy_free": False,

    # nutrition constraints
    "min_protein": None,
    "max_protein": None,
    "min_calories": None,
    "max_calories": None,
    "min_sugar": None,
    "max_sugar": None,
    "min_fat": None,
    "max_fat": None,
    "min_carbs": None,
    "max_carbs": None,
    "min_salt": None,
    "max_salt": None,

    # category-specific nutrition constraints
    "max_snack_calories": None,
    "max_drink_calories": None
}

In [9]:
#Start building the model

def solve_meal_deal(mains, snacks, drinks, params):
    model = gp.Model("tesco_meal_deal")

    # Index sets
    M = mains.index.tolist()
    S = snacks.index.tolist()
    D = drinks.index.tolist()

    # Decision variables
    x = model.addVars(M, vtype=GRB.BINARY, name="main")
    y = model.addVars(S, vtype=GRB.BINARY, name="snack")
    z = model.addVars(D, vtype=GRB.BINARY, name="drink")

    # Exactly one main, one snack, one drink
    model.addConstr(gp.quicksum(x[i] for i in M) == 1, name="one_main")
    model.addConstr(gp.quicksum(y[j] for j in S) == 1, name="one_snack")
    model.addConstr(gp.quicksum(z[k] for k in D) == 1, name="one_drink")

        # Total nutrition expressions
    total_calories = (
        gp.quicksum(mains.loc[i, "calories"] * x[i] for i in M) +
        gp.quicksum(snacks.loc[j, "calories"] * y[j] for j in S) +
        gp.quicksum(drinks.loc[k, "calories"] * z[k] for k in D)
    )

    total_protein = (
        gp.quicksum(mains.loc[i, "protein"] * x[i] for i in M) +
        gp.quicksum(snacks.loc[j, "protein"] * y[j] for j in S) +
        gp.quicksum(drinks.loc[k, "protein"] * z[k] for k in D)
    )

    total_sugar = (
        gp.quicksum(mains.loc[i, "sugar"] * x[i] for i in M) +
        gp.quicksum(snacks.loc[j, "sugar"] * y[j] for j in S) +
        gp.quicksum(drinks.loc[k, "sugar"] * z[k] for k in D)
    )

    total_fat = (
        gp.quicksum(mains.loc[i, "fat"] * x[i] for i in M) +
        gp.quicksum(snacks.loc[j, "fat"] * y[j] for j in S) +
        gp.quicksum(drinks.loc[k, "fat"] * z[k] for k in D)
    )

    total_carbs = (
        gp.quicksum(mains.loc[i, "carbs"] * x[i] for i in M) +
        gp.quicksum(snacks.loc[j, "carbs"] * y[j] for j in S) +
        gp.quicksum(drinks.loc[k, "carbs"] * z[k] for k in D)
    )

    total_salt = (
        gp.quicksum(mains.loc[i, "salt"] * x[i] for i in M) +
        gp.quicksum(snacks.loc[j, "salt"] * y[j] for j in S) +
        gp.quicksum(drinks.loc[k, "salt"] * z[k] for k in D)
    )

    total_individual_price = (
        gp.quicksum(mains.loc[i, "price"] * x[i] for i in M) +
        gp.quicksum(snacks.loc[j, "price"] * y[j] for j in S) +
        gp.quicksum(drinks.loc[k, "price"] * z[k] for k in D)
    )

    # Prime status depends ONLY on selected main
    selected_main_is_prime = gp.quicksum(mains.loc[i, "is_prime"] * x[i] for i in M)

    # Bundle price rule:
    # if selected main is prime -> 5.5
    # else -> 3.85
    meal_deal_price = (
        params["regular_price"] +
        (params["prime_price"] - params["regular_price"]) * selected_main_is_prime
    )

    total_savings = total_individual_price - meal_deal_price

        # -------------------------
    # Dietary constraints
    # -------------------------
    if params["vegetarian_only"]:
        model.addConstr(
            gp.quicksum((1 - mains.loc[i, "vegetarian"]) * x[i] for i in M) == 0,
            name="vegetarian_main"
        )
        model.addConstr(
            gp.quicksum((1 - snacks.loc[j, "vegetarian"]) * y[j] for j in S) == 0,
            name="vegetarian_snack"
        )
        model.addConstr(
            gp.quicksum((1 - drinks.loc[k, "vegetarian"]) * z[k] for k in D) == 0,
            name="vegetarian_drink"
        )

    if params["vegan_only"]:
        model.addConstr(
            gp.quicksum((1 - mains.loc[i, "vegan"]) * x[i] for i in M) == 0,
            name="vegan_main"
        )
        model.addConstr(
            gp.quicksum((1 - snacks.loc[j, "vegan"]) * y[j] for j in S) == 0,
            name="vegan_snack"
        )
        model.addConstr(
            gp.quicksum((1 - drinks.loc[k, "vegan"]) * z[k] for k in D) == 0,
            name="vegan_drink"
        )

    if params["no_pork"]:
        model.addConstr(
            gp.quicksum(mains.loc[i, "pork"] * x[i] for i in M) == 0,
            name="no_pork_main"
        )
        model.addConstr(
            gp.quicksum(snacks.loc[j, "pork"] * y[j] for j in S) == 0,
            name="no_pork_snack"
        )
        model.addConstr(
            gp.quicksum(drinks.loc[k, "pork"] * z[k] for k in D) == 0,
            name="no_pork_drink"
        )

    if params["no_beef"]:
        model.addConstr(
            gp.quicksum(mains.loc[i, "beef"] * x[i] for i in M) == 0,
            name="no_beef_main"
        )
        model.addConstr(
            gp.quicksum(snacks.loc[j, "beef"] * y[j] for j in S) == 0,
            name="no_beef_snack"
        )
        model.addConstr(
            gp.quicksum(drinks.loc[k, "beef"] * z[k] for k in D) == 0,
            name="no_beef_drink"
        )

    if params["no_chicken"]:
        model.addConstr(
            gp.quicksum(mains.loc[i, "chicken"] * x[i] for i in M) == 0,
            name="no_chicken_main"
        )
        model.addConstr(
            gp.quicksum(snacks.loc[j, "chicken"] * y[j] for j in S) == 0,
            name="no_chicken_snack"
        )
        model.addConstr(
            gp.quicksum(drinks.loc[k, "chicken"] * z[k] for k in D) == 0,
            name="no_chicken_drink"
        )

    if params["no_fish"]:
        model.addConstr(
            gp.quicksum(mains.loc[i, "fish"] * x[i] for i in M) == 0,
            name="no_fish_main"
        )
        model.addConstr(
            gp.quicksum(snacks.loc[j, "fish"] * y[j] for j in S) == 0,
            name="no_fish_snack"
        )
        model.addConstr(
            gp.quicksum(drinks.loc[k, "fish"] * z[k] for k in D) == 0,
            name="no_fish_drink"
        )

    if params["gluten_free"]:
        model.addConstr(
            gp.quicksum(mains.loc[i, "gluten"] * x[i] for i in M) == 0,
            name="gluten_free_main"
        )
        model.addConstr(
            gp.quicksum(snacks.loc[j, "gluten"] * y[j] for j in S) == 0,
            name="gluten_free_snack"
        )
        model.addConstr(
            gp.quicksum(drinks.loc[k, "gluten"] * z[k] for k in D) == 0,
            name="gluten_free_drink"
        )

    if params["dairy_free"]:
        model.addConstr(
            gp.quicksum(mains.loc[i, "dairy"] * x[i] for i in M) == 0,
            name="dairy_free_main"
        )
        model.addConstr(
            gp.quicksum(snacks.loc[j, "dairy"] * y[j] for j in S) == 0,
            name="dairy_free_snack"
        )
        model.addConstr(
            gp.quicksum(drinks.loc[k, "dairy"] * z[k] for k in D) == 0,
            name="dairy_free_drink"
        )

    # -------------------------
    # Nutrition constraints
    # -------------------------
    if params["min_protein"] is not None:
        model.addConstr(total_protein >= params["min_protein"], name="min_protein")

    if params["max_protein"] is not None:
        model.addConstr(total_protein <= params["max_protein"], name="max_protein")

    if params["min_calories"] is not None:
        model.addConstr(total_calories >= params["min_calories"], name="min_calories")

    if params["max_calories"] is not None:
        model.addConstr(total_calories <= params["max_calories"], name="max_calories")

    if params["min_sugar"] is not None:
        model.addConstr(total_sugar >= params["min_sugar"], name="min_sugar")

    if params["max_sugar"] is not None:
        model.addConstr(total_sugar <= params["max_sugar"], name="max_sugar")

    if params["min_fat"] is not None:
        model.addConstr(total_fat >= params["min_fat"], name="min_fat")

    if params["max_fat"] is not None:
        model.addConstr(total_fat <= params["max_fat"], name="max_fat")

    if params["min_carbs"] is not None:
        model.addConstr(total_carbs >= params["min_carbs"], name="min_carbs")

    if params["max_carbs"] is not None:
        model.addConstr(total_carbs <= params["max_carbs"], name="max_carbs")

    if params["min_salt"] is not None:
        model.addConstr(total_salt >= params["min_salt"], name="min_salt")

    if params["max_salt"] is not None:
        model.addConstr(total_salt <= params["max_salt"], name="max_salt")

    if params["max_snack_calories"] is not None:
        snack_calories = gp.quicksum(snacks.loc[j, "calories"] * y[j] for j in S)
        model.addConstr(snack_calories <= params["max_snack_calories"], name="max_snack_calories")

    if params["max_drink_calories"] is not None:
        drink_calories = gp.quicksum(drinks.loc[k, "calories"] * z[k] for k in D)
        model.addConstr(drink_calories <= params["max_drink_calories"], name="max_drink_calories")

        # -------------------------
    # Objective
    # -------------------------
    obj = params["objective"]

    if obj == "max_protein":
        model.setObjective(total_protein, GRB.MAXIMIZE)

    elif obj == "min_sugar":
        model.setObjective(total_sugar, GRB.MINIMIZE)

    elif obj == "min_fat":
        model.setObjective(total_fat, GRB.MINIMIZE)

    elif obj == "min_carbs":
        model.setObjective(total_carbs, GRB.MINIMIZE)

    elif obj == "min_calories":
        model.setObjective(total_calories, GRB.MINIMIZE)

    elif obj == "min_salt":
        model.setObjective(total_salt, GRB.MINIMIZE)

    elif obj == "max_savings":
        model.setObjective(total_savings, GRB.MAXIMIZE)

    else:
        raise ValueError(f"Unknown objective: {obj}")
    
        # Solve
    model.optimize()

    if model.Status != GRB.OPTIMAL:
        print("No optimal solution found.")
        print("Model status:", model.Status)
        return None

    # Extract selected items
    selected_main_idx = [i for i in M if x[i].X > 0.5][0]
    selected_snack_idx = [j for j in S if y[j].X > 0.5][0]
    selected_drink_idx = [k for k in D if z[k].X > 0.5][0]

    selected_main = mains.loc[selected_main_idx]
    selected_snack = snacks.loc[selected_snack_idx]
    selected_drink = drinks.loc[selected_drink_idx]

    result = {
        "main": selected_main["item"],
        "snack": selected_snack["item"],
        "drink": selected_drink["item"],
        "main_is_prime": int(selected_main["is_prime"]),
        "meal_deal_price": meal_deal_price.getValue(),
        "total_individual_price": total_individual_price.getValue(),
        "total_savings": total_savings.getValue(),
        "total_calories": total_calories.getValue(),
        "total_protein": total_protein.getValue(),
        "total_sugar": total_sugar.getValue(),
        "total_fat": total_fat.getValue(),
        "total_carbs": total_carbs.getValue(),
        "total_salt": total_salt.getValue(),
        "objective_used": obj,
        "objective_value": model.ObjVal
    }

    return result

In [10]:
result = solve_meal_deal(mains, snacks, drinks, params)
result

Set parameter Username
Set parameter LicenseID to value 2761760
Academic license - for non-commercial use only - expires 2027-01-07
Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (win64 - Windows 11+.0 (26200.2))

CPU model: 11th Gen Intel(R) Core(TM) i7-1165G7 @ 2.80GHz, instruction set [SSE2|AVX|AVX2|AVX512]
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 3 rows, 171 columns and 171 nonzeros (Max)
Model fingerprint: 0x07b34e10
Model has 171 linear objective coefficients and an objective constant of 3.85
Variable types: 0 continuous, 171 integer (171 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [8e-01, 5e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 1e+00]

Found heuristic solution: objective 3.9500000
Presolve removed 3 rows and 171 columns
Presolve time: 0.00s
Presolve: All rows and columns removed

Explored 0 nodes (0 simplex iterations) in 0.06 seconds (0.00 work units)

{'main': 'itsu Spicy Tuna Sushi Taco 213g',
 'snack': 'Grenade Protein Bars Salted Caramel',
 'drink': 'Emmi Cafe Latte - Caramel 370ml',
 'main_is_prime': 0,
 'meal_deal_price': 3.85,
 'total_individual_price': 9.850000000000001,
 'total_savings': 6.0,
 'total_calories': 740.0,
 'total_protein': 31.6,
 'total_sugar': 48.099999999999994,
 'total_fat': 26.799999999999997,
 'total_carbs': 95.8,
 'total_salt': 2.29,
 'objective_used': 'max_savings',
 'objective_value': 6.0}